# 03. TTL과 cache warming simulation

목표: 요청 간격, TTL과 병렬 cold start가 cache hit rate에 미치는 영향을 단순 사건 simulation으로 비교합니다. 실제 Bedrock 내부 구현을 재현하는 모델은 아닙니다.

In [ ]:
def sequential_cache(request_times, ttl_seconds):
    expires_at = None
    events = []
    for request_time in request_times:
        hit = expires_at is not None and request_time < expires_at
        events.append("read" if hit else "write")
        expires_at = request_time + ttl_seconds
    return events

times = [0, 60, 120, 240, 480, 900]
for ttl in (300, 3600):
    events = sequential_cache(times, ttl)
    hit_rate = events.count("read") / len(events)
    print(f"ttl={ttl:4d}s events={events} hit_rate={hit_rate:.1%}")

## 병렬 cold start

첫 write가 완료되기 전에 도착한 요청은 모두 miss라고 단순화합니다. Warming 호출을 먼저 완료하면 이후 병렬 요청이 read가 됩니다.

In [ ]:
def parallel_burst(arrival_times, write_duration, warmed=False):
    if warmed:
        return ["read" for _ in arrival_times]
    write_ready_at = arrival_times[0] + write_duration
    return ["write" if time < write_ready_at else "read" for time in arrival_times]

arrivals = [0.00, 0.01, 0.02, 0.05, 0.20, 0.30]
for warmed in (False, True):
    events = parallel_burst(arrivals, write_duration=0.15, warmed=warmed)
    print("warmed=", warmed, "events=", events, "hit_rate=", f"{events.count('read')/len(events):.1%}")

## TTL 선택 진단

요청 간격과 cache read/write 가상 단가를 사용해 5분과 1시간 TTL을 비교합니다. 실제 모델이 해당 TTL을 지원하는지 먼저 확인해야 합니다.

In [ ]:
def prefix_cost(events, prefix_tokens, read_price, write_price):
    units = {"read": read_price, "write": write_price}
    return sum(prefix_tokens * units[event] / 1_000_000 for event in events)

hourly_times = [0, 240, 600, 1_200, 1_800, 2_400, 3_000]
for ttl in (300, 3600):
    events = sequential_cache(hourly_times, ttl)
    cost = prefix_cost(events, 15_000, read_price=0.30, write_price=3.75)
    print(f"ttl={ttl:4d}s writes={events.count('write')} reads={events.count('read')} prefix_cost=${cost:.4f}")

## 운영 판단의 한계

긴 TTL이 항상 싸지는 않습니다. Model별 write premium, 지원 TTL, traffic 간격, cross-region routing과 cache eviction을 함께 고려해야 합니다. Simulation 예측은 실제 `cacheReadInputTokens`, `cacheWriteInputTokens`, latency와 청구액으로 검증하세요.